# TalentTrack — Atoti DataMart
**Kelompok:** Daffa Ahmad Pangreksa · Halilatunnisa · Naura Kanaya Putri M.  
**Kelas:** 2024-INT | Data Warehouse | UNESA  

Notebook ini membangun DataMart menggunakan **Atoti** dengan 3 cube OLAP:
1. `JobPostings` — analisis umum posting pekerjaan
2. `SkillDemand` — analisis permintaan skill per kategori/region/waktu
3. `SkillForecast` — hasil forecasting permintaan skill 8 minggu ke depan

---
**Prasyarat:**
```bash
pip install atoti[all] sqlalchemy psycopg2-binary python-dotenv pandas
```
Pastikan file `.env` ada di folder `talenttrack/` berisi credentials Supabase.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "atoti[jupyterlab]"], check=True)

CompletedProcess(args=['C:\\Users\\Halilatunnisa\\atoti_fix\\Scripts\\python.exe', '-m', 'pip', 'install', 'atoti[jupyterlab]'], returncode=0)

In [2]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "sqlalchemy", "psycopg2-binary"], check=True)

CompletedProcess(args=['C:\\Users\\Halilatunnisa\\atoti_fix\\Scripts\\python.exe', '-m', 'pip', 'install', 'python-dotenv', 'sqlalchemy', 'psycopg2-binary'], returncode=0)

In [3]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pip", "show", "atoti"], capture_output=True, text=True)
print(result.stdout)

Name: atoti
Version: 0.9.13
Summary: Explore metrics across hundreds of dimensions, analyze live data at its most granular level and perform what-if simulations at unparalleled speed
Home-page: 
Author: ActiveViam
Author-email: ActiveViam <atoti-dev@activeviam.com>
License: 
Location: c:\users\halilatunnisa\atoti_fix\lib\site-packages
Requires: jdk4py, atoti-server, atoti-server-parquet, atoti-client, atoti-client-parquet
Required-by: 



In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

root = Path(os.getcwd()).parent  
sys.path.insert(0, str(root))

In [5]:
load_dotenv(root / ".env", override=True)
print("PGHOST:", os.getenv("PGHOST"))
print("PGDATABASE:", os.getenv("PGDATABASE"))

PGHOST: aws-1-ap-south-1.pooler.supabase.com
PGDATABASE: postgres


In [6]:
import pandas as pd
import numpy as np
import atoti as tt
from sqlalchemy import create_engine

DB_URL = (
    f"postgresql://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT', 5432)}/{os.getenv('PGDATABASE')}"
)
engine = create_engine(DB_URL, pool_pre_ping=True)
print("DB engine created")

Welcome to Atoti 0.9.13!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.
DB engine created


## 1. Load Data dari PostgreSQL

In [7]:
from sqlalchemy import inspect
import pandas as pd

inspector = inspect(engine)

tables = inspector.get_table_names()

print("TABLES:")
for t in tables:
    print("-", t)

print("\n")

for table in tables:

    print(f"\nTABLE: {table}")

    columns = inspector.get_columns(table)

    col_data = []

    for col in columns:
        col_data.append({
            "column": col["name"],
            "type": str(col["type"])
        })

    display(pd.DataFrame(col_data))

    # sample data
    try:
        sample_df = pd.read_sql(
            f"SELECT * FROM {table} LIMIT 3",
            engine
        )

        display(sample_df)

    except Exception as e:
        print("Sample read error:", e)

TABLES:
- dim_time
- dim_location
- dim_company
- dim_position
- dim_platform
- fact_job_posting
- fact_job_posting_default
- dim_skill
- bridge_job_skill
- forecast_skill_demand
- fact_job_posting_2023
- fact_job_posting_2024
- fact_job_posting_2025
- fact_job_posting_2026



TABLE: dim_time


,column,type
0,time_id,INTEGER
1,date,DATE
2,week,INTEGER
3,month,INTEGER
4,quarter,INTEGER
5,year,INTEGER
6,week_label,TEXT


,time_id,date,week,month,quarter,year,week_label
0,1,2023-01-01,52,1,1,2023,2023-W52
1,2,2023-01-02,1,1,1,2023,2023-W01
2,3,2023-01-03,1,1,1,2023,2023-W01



TABLE: dim_location


,column,type
0,location_id,INTEGER
1,city,TEXT
2,country,TEXT
3,region,TEXT


...

,column,type
0,job_id,BIGINT
1,time_id,INTEGER
2,location_id,INTEGER
3,company_id,INTEGER
4,position_id,INTEGER
5,platform_id,INTEGER
6,posting_count,INTEGER
7,job_age_days,INTEGER
8,has_salary,BOOLEAN
9,is_remote,BOOLEAN


,job_id,time_id,location_id,company_id,position_id,platform_id,posting_count,job_age_days,has_salary,is_remote,salary_min,salary_max,source_hash,raw_title
0,152,1075,110,116,146,1,1,168,False,False,None,None,c462bb2764360aa5e861e02e41b9323b,Cleaner
1,153,866,111,117,147,1,1,377,False,False,None,None,0c4d2528ca803b8b4f0cb4c7423cc808,In Vivo Scientist
2,154,1073,110,118,148,1,1,170,False,False,None,None,f6f2af3650c58f9b52d0f15df1156022,2D Artist



TABLE: fact_job_posting_2026


,column,type
0,job_id,BIGINT
1,time_id,INTEGER
2,location_id,INTEGER
3,company_id,INTEGER
4,position_id,INTEGER
5,platform_id,INTEGER
6,posting_count,INTEGER
7,job_age_days,INTEGER
8,has_salary,BOOLEAN
9,is_remote,BOOLEAN


,job_id,time_id,location_id,company_id,position_id,platform_id,posting_count,job_age_days,has_salary,is_remote,salary_min,salary_max,source_hash,raw_title
0,151,1234,110,115,145,1,1,9,False,False,None,None,6ef2761351dee534ad0ecf2d08b67bb9,"Junior Project Manager (Remote, Contract)"
1,155,1234,110,115,149,1,1,9,False,False,None,None,c9050d4a7b8b0cab62cef861121ec739,"Middle Project Manager (Remote, Contract)"
2,156,1223,112,119,150,1,1,20,False,False,None,None,da31f38d6c50c247e9dac9e4031f6de8,E&S Senior Manager / Director - Mekong Capital


In [8]:
fact_df = pd.read_sql(
    """
    SELECT
        f.job_id,
        f.time_id,
        f.location_id,
        f.company_id,
        f.position_id,
        f.platform_id,

        f.posting_count,
        f.job_age_days,

        CASE
            WHEN f.has_salary THEN 1
            ELSE 0
        END AS has_salary,

        CASE
            WHEN f.is_remote THEN 1
            ELSE 0
        END AS is_remote,

        f.salary_min,
        f.salary_max,

        f.raw_title,
        f.source_hash,

        dt.date,
        dt.week,
        dt.month,
        dt.quarter,
        dt.year,
        dt.week_label,

        dl.city,
        dl.country,
        dl.region,

        dc.company_name,

        dp.normalized_job_title,
        dp.job_level,
        dp.job_category,

        dpl.platform_name

    FROM fact_job_posting f

    JOIN dim_time dt
        ON f.time_id = dt.time_id

    JOIN dim_location dl
        ON f.location_id = dl.location_id

    JOIN dim_company dc
        ON f.company_id = dc.company_id

    JOIN dim_position dp
        ON f.position_id = dp.position_id

    JOIN dim_platform dpl
        ON f.platform_id = dpl.platform_id
    """,
    engine
)

# remove duplicates
fact_df = fact_df.drop_duplicates(subset=["job_id"])

# datetime
fact_df["date"] = pd.to_datetime(fact_df["date"])

# numeric cleanup
numeric_cols = [
    "salary_min",
    "salary_max",
    "posting_count",
    "job_age_days"
]

for col in numeric_cols:
    fact_df[col] = pd.to_numeric(
        fact_df[col],
        errors="coerce"
    )

# fill null numeric
fact_df = fact_df.fillna({
    "salary_min": 0,
    "salary_max": 0,
    "posting_count": 0,
    "job_age_days": 0
})

# fill object nulls
for col in fact_df.select_dtypes(include="object"):
    fact_df[col] = fact_df[col].fillna("Unknown")

print(f"fact_job_posting : {len(fact_df):,} rows")

fact_df.head()

fact_job_posting : 1,956 rows


,job_id,time_id,location_id,company_id,position_id,platform_id,posting_count,job_age_days,has_salary,is_remote,...,year,week_label,city,country,region,company_name,normalized_job_title,job_level,job_category,platform_name
0,1,473,1,1,1,1,1,770,0,0,...,2024,2024-W16,Princeton,United States,North America,Corcoran Sawyer Smith,marketing coordinator,Manager,Management,LinkedIn
1,2,467,2,2,2,1,1,776,1,0,...,2024,2024-W15,Fort Collins,United States,North America,Unknown,mental health therapist/counselor,Unknown,Healthcare,LinkedIn
2,3,472,3,3,3,1,1,771,1,0,...,2024,2024-W16,Cincinnati,United States,North America,The National Exemplar,assitant restaurant manager,Manager,Management,LinkedIn
3,4,468,4,4,4,1,1,775,1,0,...,2024,2024-W15,New Hyde Park,United States,North America,"Abrams Fensterman, Llp",senior elder law / trusts and estates associat...,Junior,Legal,LinkedIn
4,5,474,5,2,5,1,1,769,1,0,...,2024,2024-W16,Burlington,United States,North America,Unknown,service technician,Unknown,Other,LinkedIn


In [9]:
skill_df = pd.read_sql(
    """
    SELECT
        b.job_id,
        b.skill_id,

        b.extraction_confidence,

        ds.skill_name,
        ds.skill_type,
        ds.skill_domain

    FROM bridge_job_skill b

    JOIN dim_skill ds
        ON b.skill_id = ds.skill_id
    """,
    engine
)

# numeric cleanup
skill_df["extraction_confidence"] = pd.to_numeric(
    skill_df["extraction_confidence"],
    errors="coerce"
)

# fill null confidence
skill_df["extraction_confidence"] = (
    skill_df["extraction_confidence"]
    .fillna(0)
)

# remove duplicates
skill_df = skill_df.drop_duplicates()

# fill object nulls
for col in skill_df.select_dtypes(include="object"):
    skill_df[col] = skill_df[col].fillna("Unknown")

print(f"bridge_job_skill + dim_skill : {len(skill_df):,} rows")

skill_df.head(3)

bridge_job_skill + dim_skill : 6,544 rows


,job_id,skill_id,extraction_confidence,skill_name,skill_type,skill_domain
0,1,1,0.3922,graphic design,Knowledge,urchade/gliner_small-v2.1
1,1,2,0.6282,adobe creative cloud,Knowledge,urchade/gliner_small-v2.1
2,1,3,0.3554,indesign,Knowledge,urchade/gliner_small-v2.1


In [10]:
forecast_df = pd.read_sql(
    """
    SELECT
        fsd.forecast_id,

        fsd.skill_id,

        ds.skill_name,
        ds.skill_type,
        ds.skill_domain,

        fsd.job_category,

        fsd.forecast_week_label,
        fsd.forecast_year,
        fsd.forecast_week,

        fsd.predicted_count,
        fsd.lower_bound,
        fsd.upper_bound,
        fsd.trend_score,

        fsd.model_name,
        fsd.generated_at

    FROM forecast_skill_demand fsd

    JOIN dim_skill ds
        ON fsd.skill_id = ds.skill_id
    """,
    engine
)

# numeric cleanup
numeric_cols = [
    "predicted_count",
    "lower_bound",
    "upper_bound",
    "trend_score"
]

for col in numeric_cols:
    forecast_df[col] = pd.to_numeric(
        forecast_df[col],
        errors="coerce"
    )

# fill null numeric
forecast_df = forecast_df.fillna({
    "predicted_count": 0,
    "lower_bound": 0,
    "upper_bound": 0,
    "trend_score": 0
})

# datetime cleanup
forecast_df["generated_at"] = pd.to_datetime(
    forecast_df["generated_at"],
    errors="coerce"
)

# fill object nulls
for col in forecast_df.select_dtypes(include="object"):
    forecast_df[col] = forecast_df[col].fillna("Unknown")

# remove duplicates
forecast_df = forecast_df.drop_duplicates()

print(f"forecast_skill_demand : {len(forecast_df):,} rows")

forecast_df.head(3)

forecast_skill_demand : 26,560 rows


,forecast_id,skill_id,skill_name,skill_type,skill_domain,job_category,forecast_week_label,forecast_year,forecast_week,predicted_count,lower_bound,upper_bound,trend_score,model_name,generated_at
0,159553,4103,2-d cad methods,Knowledge,urchade/gliner_small-v2.1,Management,2024-W17,2024,17,1.0,1.0,1.0,0.0,mean,2026-05-27 17:11:56.182558+00:00
1,3161,881,1-5 years of experiencebachelor s degree,Skill,urchade/gliner_small-v2.1,Management,2025-W49,2025,49,1.0,1.0,1.0,0.0,mean,2026-05-27 17:11:56.182558+00:00
2,3162,881,1-5 years of experiencebachelor s degree,Skill,urchade/gliner_small-v2.1,Management,2025-W50,2025,50,1.0,1.0,1.0,0.0,mean,2026-05-27 17:11:56.182558+00:00


In [11]:
skill_fact_df = skill_df.merge(
    fact_df,
    on="job_id",
    how="left"
)

# numeric cleanup
skill_fact_df["salary_min"] = pd.to_numeric(
    skill_fact_df["salary_min"],
    errors="coerce"
)

skill_fact_df["salary_max"] = pd.to_numeric(
    skill_fact_df["salary_max"],
    errors="coerce"
)

# safer midpoint calculation
skill_fact_df["salary_midpoint"] = (
    skill_fact_df[
        ["salary_min", "salary_max"]
    ]
    .mean(axis=1)
)

# fill missing midpoint
skill_fact_df["salary_midpoint"] = (
    skill_fact_df["salary_midpoint"]
    .fillna(0)
)

# measure helper
skill_fact_df["skill_count"] = 1

# remove duplicate job-skill rows
skill_fact_df = skill_fact_df.drop_duplicates(
    subset=["job_id", "skill_id"]
)

# fill object nulls
for col in skill_fact_df.select_dtypes(include="object"):
    skill_fact_df[col] = (
        skill_fact_df[col]
        .fillna("Unknown")
    )

print(f"skill_fact_df merged : {len(skill_fact_df):,} rows")

skill_fact_df.head(3)

skill_fact_df merged : 6,544 rows


,job_id,skill_id,extraction_confidence,skill_name,skill_type,skill_domain,time_id,location_id,company_id,position_id,...,city,country,region,company_name,normalized_job_title,job_level,job_category,platform_name,salary_midpoint,skill_count
0,1,1,0.3922,graphic design,Knowledge,urchade/gliner_small-v2.1,473,1,1,1,...,Princeton,United States,North America,Corcoran Sawyer Smith,marketing coordinator,Manager,Management,LinkedIn,0.0,1
1,1,2,0.6282,adobe creative cloud,Knowledge,urchade/gliner_small-v2.1,473,1,1,1,...,Princeton,United States,North America,Corcoran Sawyer Smith,marketing coordinator,Manager,Management,LinkedIn,0.0,1
2,1,3,0.3554,indesign,Knowledge,urchade/gliner_small-v2.1,473,1,1,1,...,Princeton,United States,North America,Corcoran Sawyer Smith,marketing coordinator,Manager,Management,LinkedIn,0.0,1


## 2. Buat Sesi Atoti

In [12]:
# fact_df cleanup
fact_df["salary_min"] = (
    fact_df["salary_min"]
    .fillna(0)
)

fact_df["salary_max"] = (
    fact_df["salary_max"]
    .fillna(0)
)

fact_df["city"] = (
    fact_df["city"]
    .fillna("Unknown")
)

fact_df["country"] = (
    fact_df["country"]
    .fillna("Unknown")
)

fact_df["region"] = (
    fact_df["region"]
    .fillna("Unknown")
)

fact_df["company_name"] = (
    fact_df["company_name"]
    .fillna("Unknown")
)

fact_df["job_category"] = (
    fact_df["job_category"]
    .fillna("Unknown")
)

fact_df["job_level"] = (
    fact_df["job_level"]
    .fillna("Unknown")
)

fact_df["normalized_job_title"] = (
    fact_df["normalized_job_title"]
    .fillna("Unknown")
)

fact_df["platform_name"] = (
    fact_df["platform_name"]
    .fillna("Unknown")
)

# skill_fact_df cleanup
str_cols = [
    "skill_domain",
    "skill_type",
    "skill_name",

    "job_category",
    "job_level",
    "normalized_job_title",

    "region",
    "country",
    "city",

    "week_label",
    "platform_name",
    "company_name"
]

for col in str_cols:

    if col in skill_fact_df.columns:

        skill_fact_df[col] = (
            skill_fact_df[col]
            .fillna("Unknown")
        )

# integer columns
for col in [
    "year",
    "quarter",
    "month",
    "week"
]:

    if col in skill_fact_df.columns:

        skill_fact_df[col] = (
            skill_fact_df[col]
            .fillna(0)
            .astype(int)
        )

# numeric cleanup
skill_fact_df["extraction_confidence"] = (
    skill_fact_df["extraction_confidence"]
    .fillna(0.0)
)

skill_fact_df["salary_midpoint"] = (
    skill_fact_df["salary_midpoint"]
    .fillna(0.0)
)

# forecast_df cleanup
forecast_str_cols = [
    "skill_name",
    "skill_type",
    "skill_domain",
    "job_category",
    "forecast_week_label",
    "model_name"
]

for col in forecast_str_cols:

    if col in forecast_df.columns:

        forecast_df[col] = (
            forecast_df[col]
            .fillna("Unknown")
        )

forecast_df["forecast_year"] = (
    forecast_df["forecast_year"]
    .fillna(0)
    .astype(int)
)

forecast_df["forecast_week"] = (
    forecast_df["forecast_week"]
    .fillna(0)
    .astype(int)
)

forecast_numeric_cols = [
    "predicted_count",
    "lower_bound",
    "upper_bound",
    "trend_score"
]

for col in forecast_numeric_cols:

    if col in forecast_df.columns:

        forecast_df[col] = (
            forecast_df[col]
            .fillna(0.0)
        )

print("Fillna cleanup done")

Fillna cleanup done


In [13]:
session = tt.Session.start(
    tt.SessionConfig(
        port=9090,
        user_content_storage="./content"
    )
)

print("Atoti session started")
print("Dashboard URL:", session.url)

Atoti session started
Dashboard URL: http://localhost:9090


## 3. Cube 1: JobPostings
Analisis umum: volume posting, remote rate, salary coverage, per platform/region/waktu.

In [14]:
fact_df["is_remote"] = fact_df["is_remote"].map({
    1: "Remote",
    0: "Onsite",
    True: "Remote",
    False: "Onsite"
})

fact_df["has_salary"] = fact_df["has_salary"].map({
    1: "With Salary",
    0: "No Salary",
    True: "With Salary",
    False: "No Salary"
})

job_store = session.read_pandas(
    fact_df,

    table_name="job_postings",

    keys=["job_id"],

    default_values={

        "time_id": 0,
        "location_id": 0,
        "company_id": 0,
        "position_id": 0,
        "platform_id": 0,

        "year": 0,
        "quarter": 0,
        "month": 0,
        "week": 0,

        "week_label": "Unknown",

        "city": "Unknown",
        "country": "Unknown",
        "region": "Unknown",

        "company_name": "Unknown",

        "normalized_job_title": "Unknown",
        "job_level": "Unknown",
        "job_category": "Unknown",

        "platform_name": "Unknown",

        "salary_min": 0.0,
        "salary_max": 0.0,

        "posting_count": 0,
        "job_age_days": 0,

        "has_salary": "Unknown",
        "is_remote": "Unknown",
    }
)

# CREATE CUBE
job_cube = session.create_cube(
    job_store,
    name="JobPostings"
)

h = job_cube.hierarchies
m = job_cube.measures

h["Time"] = [
    job_store["year"],
    job_store["quarter"],
    job_store["month"],
    job_store["week_label"]
]

h["Location"] = [
    job_store["region"],
    job_store["country"],
    job_store["city"]
]

h["Job"] = [
    job_store["job_category"],
    job_store["job_level"],
    job_store["normalized_job_title"]
]

h["Company"] = [
    job_store["company_name"]
]

h["Platform"] = [
    job_store["platform_name"]
]

h["Remote Status"] = [
    job_store["is_remote"]
]

h["Salary Status"] = [
    job_store["has_salary"]
]

m["Total Postings"] = tt.agg.sum(
    job_store["posting_count"]
)

m["Avg Salary Min"] = tt.agg.mean(
    job_store["salary_min"]
)

m["Avg Salary Max"] = tt.agg.mean(
    job_store["salary_max"]
)

m["Avg Job Age Days"] = tt.agg.mean(
    job_store["job_age_days"]
)

print("Cube 1 (JobPostings) created")

Cube 1 (JobPostings) created


## 4. Cube 2: SkillDemand
Analisis permintaan skill: skill apa paling banyak diminta, per kategori, region, waktu.

In [15]:
skill_store = session.read_pandas(
    skill_fact_df,

    table_name="skill_facts",

    keys=[
        "job_id",
        "skill_id"
    ],

    default_values={

        "skill_id": 0,

        "skill_domain": "Unknown",
        "skill_type": "Unknown",
        "skill_name": "Unknown",

        "job_category": "Unknown",
        "job_level": "Unknown",
        "normalized_job_title": "Unknown",

        "region": "Unknown",
        "country": "Unknown",
        "city": "Unknown",

        "year": 0,
        "quarter": 0,
        "month": 0,
        "week": 0,
        "week_label": "Unknown",

        "platform_name": "Unknown",

        "company_name": "Unknown",

        "extraction_confidence": 0.0,
        "salary_midpoint": 0.0,

        "skill_count": 0,
    }
)

# CREATE CUBE
skill_cube = session.create_cube(
    skill_store,
    name="SkillDemand"
)

sh = skill_cube.hierarchies
sm = skill_cube.measures

# HIERARCHIES
sh["Skill"] = [
    skill_store["skill_domain"],
    skill_store["skill_type"],
    skill_store["skill_name"]
]

sh["Job"] = [
    skill_store["job_category"],
    skill_store["job_level"],
    skill_store["normalized_job_title"]
]

sh["Location"] = [
    skill_store["region"],
    skill_store["country"],
    skill_store["city"]
]

sh["Time"] = [
    skill_store["year"],
    skill_store["quarter"],
    skill_store["month"],
    skill_store["week_label"]
]

sh["Platform"] = [
    skill_store["platform_name"]
]

sh["Company"] = [
    skill_store["company_name"]
]

# MEASURES
sm["Skill Postings"] = tt.agg.sum(
    skill_store["skill_count"]
)

sm["Avg Confidence"] = tt.agg.mean(
    skill_store["extraction_confidence"]
)

sm["Avg Salary Midpoint"] = tt.agg.mean(
    skill_store["salary_midpoint"]
)

print("Cube 2 (SkillDemand) created")

Cube 2 (SkillDemand) created


## 5. Cube 3: SkillForecast
Hasil forecasting Holt-Winters/Linear Trend: prediksi permintaan skill 8 minggu ke depan.

In [16]:
forecast_store = session.read_pandas(
    forecast_df,

    table_name="skill_forecasts",

    keys=[
        "forecast_id"
    ],

    default_values={

        "forecast_id": 0,

        "skill_id": 0,

        "skill_name": "Unknown",
        "skill_type": "Unknown",
        "skill_domain": "Unknown",

        "job_category": "Unknown",

        "forecast_year": 0,
        "forecast_week": 0,
        "forecast_week_label": "Unknown",

        "model_name": "Unknown",

        "predicted_count": 0.0,
        "lower_bound": 0.0,
        "upper_bound": 0.0,
        "trend_score": 0.0,
    }
)

# CREATE CUBE
forecast_cube = session.create_cube(
    forecast_store,
    name="SkillForecast"
)

fh = forecast_cube.hierarchies
fm = forecast_cube.measures

# HIERARCHIES
fh["Skill"] = [
    forecast_store["skill_domain"],
    forecast_store["skill_type"],
    forecast_store["skill_name"]
]

fh["Job"] = [
    forecast_store["job_category"]
]

fh["Forecast Time"] = [
    forecast_store["forecast_year"],
    forecast_store["forecast_week"],
    forecast_store["forecast_week_label"]
]

fh["Model"] = [
    forecast_store["model_name"]
]

# MEASURES
fm["Predicted Postings"] = tt.agg.sum(
    forecast_store["predicted_count"]
)

fm["Forecast Lower"] = tt.agg.sum(
    forecast_store["lower_bound"]
)

fm["Forecast Upper"] = tt.agg.sum(
    forecast_store["upper_bound"]
)

fm["Forecast Range"] = (
    fm["Forecast Upper"]
    - fm["Forecast Lower"]
)

fm["Trend Score"] = tt.agg.mean(
    forecast_store["trend_score"]
)

print("Cube 3 (SkillForecast) created")

Cube 3 (SkillForecast) created


In [17]:
# LOAD DIMENSION TABLES
dim_time_df = pd.read_sql(
    "SELECT * FROM dim_time",
    engine
)

dim_location_df = pd.read_sql(
    "SELECT * FROM dim_location",
    engine
)

dim_company_df = pd.read_sql(
    "SELECT * FROM dim_company",
    engine
)

dim_position_df = pd.read_sql(
    "SELECT * FROM dim_position",
    engine
)

dim_platform_df = pd.read_sql(
    "SELECT * FROM dim_platform",
    engine
)

dim_skill_df = pd.read_sql(
    "SELECT * FROM dim_skill",
    engine
)

# DATETIME CLEANUP
dim_time_df["date"] = pd.to_datetime(
    dim_time_df["date"],
    errors="coerce"
)

# CLEANUP FUNCTION
dimension_dfs = [
    dim_time_df,
    dim_location_df,
    dim_company_df,
    dim_position_df,
    dim_platform_df,
    dim_skill_df
]

for df in dimension_dfs:

    # fill object nulls
    for col in df.select_dtypes(include="object").columns:

        df[col] = (
            df[col]
            .fillna("Unknown")
        )

    # fill numeric nulls
    for col in df.select_dtypes(include="number").columns:

        df[col] = (
            df[col]
            .fillna(0)
        )

# REMOVE DUPLICATES
dim_time_df = dim_time_df.drop_duplicates()
dim_location_df = dim_location_df.drop_duplicates()
dim_company_df = dim_company_df.drop_duplicates()
dim_position_df = dim_position_df.drop_duplicates()
dim_platform_df = dim_platform_df.drop_duplicates()
dim_skill_df = dim_skill_df.drop_duplicates()

print("Dimension tables loaded successfully")

Dimension tables loaded successfully


In [18]:
# LOAD DIMENSION STORES
dim_time_store = session.read_pandas(
    dim_time_df,
    table_name="dim_time",
    keys=["time_id"]
)

dim_location_store = session.read_pandas(
    dim_location_df,
    table_name="dim_location",
    keys=["location_id"]
)

dim_company_store = session.read_pandas(
    dim_company_df,
    table_name="dim_company",
    keys=["company_id"]
)

dim_position_store = session.read_pandas(
    dim_position_df,
    table_name="dim_position",
    keys=["position_id"]
)

dim_platform_store = session.read_pandas(
    dim_platform_df,
    table_name="dim_platform",
    keys=["platform_id"]
)

dim_skill_store = session.read_pandas(
    dim_skill_df,
    table_name="dim_skill",
    keys=["skill_id"]
)

print("Dimension stores loaded")

# STAR SCHEMA RELATIONSHIPS
# fact_job_posting -> dimensions
job_store.join(
    dim_time_store,
    job_store["time_id"] == dim_time_store["time_id"]
)

job_store.join(
    dim_location_store,
    job_store["location_id"] == dim_location_store["location_id"]
)

job_store.join(
    dim_company_store,
    job_store["company_id"] == dim_company_store["company_id"]
)

job_store.join(
    dim_position_store,
    job_store["position_id"] == dim_position_store["position_id"]
)

job_store.join(
    dim_platform_store,
    job_store["platform_id"] == dim_platform_store["platform_id"]
)

# skill_fact -> skill dimension
skill_store.join(
    dim_skill_store,
    skill_store["skill_id"] == dim_skill_store["skill_id"]
)

print("Relationships created")

Dimension stores loaded
Relationships created


In [19]:
session.tables.schema

```mermaid
erDiagram
  "job_postings" {
    non-null long PK "job_id"
    non-null long "time_id"
    non-null long "location_id"
    non-null long "company_id"
    non-null long "position_id"
    non-null long "platform_id"
    non-null long "posting_count"
    non-null long "job_age_days"
    non-null String "has_salary"
    non-null String "is_remote"
    non-null double "salary_min"
    non-null double "salary_max"
    non-null String "raw_title"
    non-null String "source_hash"
    non-null LocalDate "date"
    non-null long "week"
    non-null long "month"
    non-null long "quarter"
    non-null long "year"
    non-null String "week_label"
    non-null String "city"
    non-null String "country"
    non-null String "region"
    non-null String "company_name"
    non-null String "normalized_job_title"
    non-null String "job_level"
    non-null String "job_category"
    non-null String "platform_name"
  }
  "skill_facts" {
    non-null long PK "job_id"
    non-null long PK "skill_id"
    non-null double "extraction_confidence"
    non-null String "skill_name"
    non-null String "skill_type"
    non-null String "skill_domain"
    nullable long "time_id"
    nullable long "location_id"
    nullable long "company_id"
    nullable long "position_id"
    nullable long "platform_id"
    nullable long "posting_count"
    nullable long "job_age_days"
    nullable long "has_salary"
    nullable long "is_remote"
    nullable double "salary_min"
    nullable double "salary_max"
    non-null String "raw_title"
    non-null String "source_hash"
    non-null LocalDate "date"
    non-null long "week"
    non-null long "month"
    non-null long "quarter"
    non-null long "year"
    non-null String "week_label"
    non-null String "city"
    non-null String "country"
    non-null String "region"
    non-null String "company_name"
    non-null String "normalized_job_title"
    non-null String "job_level"
    non-null String "job_category"
    non-null String "platform_name"
    non-null double "salary_midpoint"
    non-null long "skill_count"
  }
  "skill_forecasts" {
    non-null long PK "forecast_id"
    non-null long "skill_id"
    non-null String "skill_name"
    non-null String "skill_type"
    non-null String "skill_domain"
    non-null String "job_category"
    non-null String "forecast_week_label"
    non-null long "forecast_year"
    non-null long "forecast_week"
    non-null double "predicted_count"
    non-null double "lower_bound"
    non-null double "upper_bound"
    non-null double "trend_score"
    non-null String "model_name"
    non-null ZonedDateTime "generated_at"
  }
  "dim_time" {
    non-null long PK "time_id"
    non-null LocalDate "date"
    nullable long "week"
    nullable long "month"
    nullable long "quarter"
    nullable long "year"
    non-null String "week_label"
  }
  "dim_location" {
    non-null long PK "location_id"
    non-null String "city"
    non-null String "country"
    non-null String "region"
  }
  "dim_company" {
    non-null long PK "company_id"
    non-null String "company_name"
  }
  "dim_position" {
    non-null long PK "position_id"
    non-null String "normalized_job_title"
    non-null String "job_level"
    non-null String "job_category"
  }
  "dim_platform" {
    non-null long PK "platform_id"
    non-null String "platform_name"
  }
  "dim_skill" {
    non-null long PK "skill_id"
    non-null String "skill_name"
    non-null String "skill_type"
    non-null String "skill_domain"
  }
  "job_postings" }o--o| "dim_company" : "company_id == company_id"
  "job_postings" }o--o| "dim_location" : "location_id == location_id"
  "job_postings" }o--o| "dim_platform" : "platform_id == platform_id"
  "job_postings" }o--o| "dim_position" : "position_id == position_id"
  "job_postings" }o--o| "dim_time" : "time_id == time_id"
  "skill_facts" }o--o| "dim_skill" : "skill_id == skill_id"
```


## 6. Widgets — Insight 1: Top 20 Skill Paling Banyak Diminta

In [20]:
top_skills = (
    skill_cube.query(
        skill_cube.measures["Skill Postings"],

        levels=[
            skill_cube.hierarchies["Skill"]["skill_name"]
        ],

        include_totals=False
    )

    .reset_index()

    .query("skill_name != 'Unknown'")

    .sort_values(
        by="Skill Postings",
        ascending=False
    )

    .head(20)
)

print("Top 20 Skills by Posting Count")

top_skills

Top 20 Skills by Posting Count


,skill_domain,skill_type,skill_name,Skill Postings
3386,urchade/gliner_small-v2.1,Skill,english,72
294,urchade/gliner_small-v2.1,Knowledge,bachelor s degree,62
3877,urchade/gliner_small-v2.1,Skill,python,38
1625,urchade/gliner_small-v2.1,Knowledge,knowledge,37
4007,urchade/gliner_small-v2.1,Skill,sql,31
1310,urchade/gliner_small-v2.1,Knowledge,grade 12,31
1391,urchade/gliner_small-v2.1,Knowledge,high school graduate,28
3842,urchade/gliner_small-v2.1,Skill,professional,26
1858,urchade/gliner_small-v2.1,Knowledge,microsoft,26
3861,urchade/gliner_small-v2.1,Skill,progressive,24


## 7. Widgets — Insight 2: Tren Skill per Minggu

In [21]:
skill_trend = (
    skill_cube.query(
        skill_cube.measures["Skill Postings"],

        levels=[
            skill_cube.hierarchies["Time"]["week_label"],
            skill_cube.hierarchies["Skill"]["skill_name"],
        ],

        include_totals=False,
    )

    .reset_index()

    .query("skill_name != 'Unknown'")

    .sort_values(
        by=["week_label", "Skill Postings"],
        ascending=[True, False]
    )
)

print(f"Skill trend shape: {skill_trend.shape}")

skill_trend.head(10)

Skill trend shape: (5168, 8)


,year,quarter,month,week_label,skill_domain,skill_type,skill_name,Skill Postings
109,2024,2,4,2024-W14,urchade/gliner_small-v2.1,Knowledge,bachelor s degree,18
537,2024,2,4,2024-W14,urchade/gliner_small-v2.1,Knowledge,knowledge,17
1238,2024,2,4,2024-W14,urchade/gliner_small-v2.1,Skill,python,14
1272,2024,2,4,2024-W14,urchade/gliner_small-v2.1,Skill,sql,13
343,2024,2,4,2024-W14,urchade/gliner_small-v2.1,Knowledge,equipment,11
88,2024,2,4,2024-W14,urchade/gliner_small-v2.1,Knowledge,aws,9
90,2024,2,4,2024-W14,urchade/gliner_small-v2.1,Knowledge,azure,7
598,2024,2,4,2024-W14,urchade/gliner_small-v2.1,Knowledge,microsoft,7
829,2024,2,4,2024-W14,urchade/gliner_small-v2.1,Knowledge,salesforce,7
1046,2024,2,4,2024-W14,urchade/gliner_small-v2.1,Skill,attention to detail,7


## 8. Widgets — Insight 3: Perbandingan Platform

In [22]:
# PLATFORM ANALYSIS FROM ATOTI
platform_atoti = (
    job_cube.query(

        job_cube.measures["Total Postings"],
        job_cube.measures["Avg Salary Max"],

        levels=[
            job_cube.hierarchies["Platform"]["platform_name"],
            job_cube.hierarchies["Job"]["job_category"],
            job_cube.hierarchies["Remote Status"]["is_remote"],
            job_cube.hierarchies["Salary Status"]["has_salary"],
        ],

        include_totals=False,
    )

    .reset_index()

    .sort_values(
        by="Total Postings",
        ascending=False
    )
)

print(f"Platform comparison (Atoti): {platform_atoti.shape}")

platform_atoti.head(10)

Platform comparison (Atoti): (103, 6)


,platform_name,job_category,is_remote,has_salary,Total Postings,Avg Salary Max
79,LinkedIn,Other,Onsite,No Salary,558,0.0
60,LinkedIn,Management,Onsite,No Salary,244,0.0
80,LinkedIn,Other,Onsite,With Salary,126,89802.957698
61,LinkedIn,Management,Onsite,With Salary,100,106435.204
0,LinkedIn,Administration,Onsite,No Salary,58,0.0
90,LinkedIn,Sales,Onsite,No Salary,58,0.0
81,LinkedIn,Other,Remote,No Salary,51,0.0
18,LinkedIn,Data,Onsite,No Salary,48,0.0
62,LinkedIn,Management,Remote,No Salary,35,0.0
32,LinkedIn,Engineering,Onsite,No Salary,35,0.0


## 9. Widgets — Insight 4: Forecasting Skill 8 Minggu ke Depan

In [23]:
forecast_result = (
    forecast_cube.query(
        forecast_cube.measures["Predicted Postings"],
        forecast_cube.measures["Forecast Lower"],
        forecast_cube.measures["Forecast Upper"],

        levels=[
            forecast_cube.hierarchies["Forecast Time"]["forecast_week_label"],
            forecast_cube.hierarchies["Skill"]["skill_name"],
        ],

        include_totals=False,
    )

    .reset_index()

    .query("skill_name != 'Unknown'")

    .sort_values(
        by="Predicted Postings",
        ascending=False
    )
)

print(f"Forecast result shape: {forecast_result.shape}")

forecast_result.head(15)

Forecast result shape: (23104, 9)


,forecast_year,forecast_week,forecast_week_label,skill_domain,skill_type,skill_name,Predicted Postings,Forecast Lower,Forecast Upper
3935,2024,18,2024-W18,urchade/gliner_small-v2.1,Knowledge,bachelor s degree,13.995,13.015,14.975
2108,2024,17,2024-W17,urchade/gliner_small-v2.1,Knowledge,bachelor s degree,13.995,13.015,14.975
7589,2024,20,2024-W20,urchade/gliner_small-v2.1,Knowledge,bachelor s degree,13.995,13.015,14.975
9416,2024,21,2024-W21,urchade/gliner_small-v2.1,Knowledge,bachelor s degree,13.995,13.015,14.975
5762,2024,19,2024-W19,urchade/gliner_small-v2.1,Knowledge,bachelor s degree,13.995,13.015,14.975
11243,2024,22,2024-W22,urchade/gliner_small-v2.1,Knowledge,bachelor s degree,13.995,13.015,14.975
12755,2024,22,2024-W22,urchade/gliner_small-v2.1,Skill,python,12.995,12.015,13.975
7274,2024,19,2024-W19,urchade/gliner_small-v2.1,Skill,python,12.995,12.015,13.975
9101,2024,20,2024-W20,urchade/gliner_small-v2.1,Skill,python,12.995,12.015,13.975
3620,2024,17,2024-W17,urchade/gliner_small-v2.1,Skill,python,12.995,12.015,13.975


## 10. Widgets — Insight 5: Salary per Skill dan Job Level

In [24]:
salary_by_skill = (
    skill_cube.query(
        skill_cube.measures["Avg Salary Midpoint"],
        skill_cube.measures["Skill Postings"],

        levels=[
            skill_cube.hierarchies["Skill"]["skill_name"],
            skill_cube.hierarchies["Job"]["job_level"],
        ],

        include_totals=False,
    )

    .reset_index()

    .query("skill_name != 'Unknown'")

    .query("`Avg Salary Midpoint` > 0")

    .sort_values(
        by="Avg Salary Midpoint",
        ascending=False
    )
)

print(f"Salary by skill/level: {salary_by_skill.shape}")

salary_by_skill.head(15)

Salary by skill/level: (952, 7)


,skill_domain,skill_type,skill_name,job_category,job_level,Avg Salary Midpoint,Skill Postings
807,urchade/gliner_small-v2.1,Knowledge,comprehensive expertise,Legal,Unknown,427500.0,1
3007,urchade/gliner_small-v2.1,Knowledge,relevant experience,Legal,Unknown,427500.0,1
2979,urchade/gliner_small-v2.1,Knowledge,real estate license,Sales,Unknown,400000.0,1
4383,urchade/gliner_small-v2.1,Skill,excellent communication and interpersonal skills,Legal,Unknown,317200.0,1
2023,urchade/gliner_small-v2.1,Knowledge,juris doctor,Legal,Unknown,317200.0,1
3050,urchade/gliner_small-v2.1,Knowledge,riteload,Sales,Senior,300000.0,1
1022,urchade/gliner_small-v2.1,Knowledge,dea certificationvalid,Other,Senior,286000.0,1
531,urchade/gliner_small-v2.1,Knowledge,board certification,Other,Senior,286000.0,1
127,urchade/gliner_small-v2.1,Knowledge,ai,Finance,Unknown,255000.0,1
2517,urchade/gliner_small-v2.1,Knowledge,nile,Finance,Unknown,255000.0,1


## 11. Widgets — Insight 6: Distribusi Geografis Posting

In [25]:
geo_distribution = (
    job_cube.query(
        job_cube.measures["Total Postings"],

        levels=[
            job_cube.hierarchies["Location"]["region"],
            job_cube.hierarchies["Location"]["country"],
            job_cube.hierarchies["Job"]["job_category"],
            job_cube.hierarchies["Remote Status"]["is_remote"],  
        ],

        include_totals=True,
    )

    .reset_index()

    .query("country != 'Unknown'")
    .sort_values(
        by="Total Postings",
        ascending=False
    )
)

## 12. Widgets — Insight 7: Top Skill per Job Category (dari MV)

In [26]:
top_skills_per_cat = pd.read_sql(
    """
    SELECT
        job_category,
        skill_name,
        skill_domain,

        SUM(posting_count) AS total_postings

    FROM mv_weekly_skill_demand

    GROUP BY
        job_category,
        skill_name,
        skill_domain

    ORDER BY
        job_category,
        total_postings DESC

    LIMIT 200
    """,
    engine
)

top5_per_cat = (
    top_skills_per_cat

    .query("skill_name != 'Unknown'")

    .sort_values(
        by=["job_category", "total_postings"],
        ascending=[True, False]
    )

    .groupby("job_category")

    .head(5)

    .reset_index(drop=True)
)

print(f"Top 5 skill per category: {len(top5_per_cat):,} rows")

top5_per_cat

Top 5 skill per category: 10 rows


,job_category,skill_name,skill_domain,total_postings
0,Administration,microsoft,urchade/gliner_small-v2.1,6.0
1,Administration,strong written and verbal communication skills,urchade/gliner_small-v2.1,3.0
2,Administration,clerical skills,urchade/gliner_small-v2.1,2.0
3,Administration,netflix,urchade/gliner_small-v2.1,2.0
4,Administration,linux,urchade/gliner_small-v2.1,2.0
5,Cloud,azure,urchade/gliner_small-v2.1,2.0
6,Cloud,mcse,urchade/gliner_small-v2.1,1.0
7,Cloud,mcsd,urchade/gliner_small-v2.1,1.0
8,Cloud,cloudformation,urchade/gliner_small-v2.1,1.0
9,Cloud,dockerexperience,urchade/gliner_small-v2.1,1.0


## 12. OLAP CUBE Query (PostgreSQL)

In [27]:
cube_query = pd.read_sql(
    """
    SELECT
        dt.year,
        dt.quarter,

        ds.skill_name,
        ds.skill_domain,

        dp.job_category,

        dl.region,

        COUNT(DISTINCT f.job_id) AS posting_count,

        AVG(f.salary_max) AS avg_salary_max

    FROM fact_job_posting f

    JOIN bridge_job_skill b
        ON f.job_id = b.job_id

    JOIN dim_skill ds
        ON b.skill_id = ds.skill_id

    JOIN dim_time dt
        ON f.time_id = dt.time_id

    JOIN dim_position dp
        ON f.position_id = dp.position_id

    JOIN dim_location dl
        ON f.location_id = dl.location_id

    GROUP BY CUBE(
        dt.year,
        dt.quarter,
        ds.skill_name,
        ds.skill_domain,
        dp.job_category,
        dl.region
    )

    HAVING COUNT(DISTINCT f.job_id) > 0

    ORDER BY
        dt.year NULLS LAST,
        posting_count DESC

    LIMIT 1000
    """,
    engine
)

# CLEAN RESULT
cube_query["skill_name"] = cube_query["skill_name"].fillna("ALL")
cube_query["skill_domain"] = cube_query["skill_domain"].fillna("ALL")
cube_query["job_category"] = cube_query["job_category"].fillna("ALL")
cube_query["region"] = cube_query["region"].fillna("ALL")

cube_query["avg_salary_max"] = (
    pd.to_numeric(
        cube_query["avg_salary_max"],
        errors="coerce"
    )
    .fillna(0)
)

cube_query = cube_query.sort_values(
    by="posting_count",
    ascending=False
)

print(f"CUBE query: {len(cube_query):,} rows")

cube_query.head(10)

CUBE query: 1,000 rows


,year,quarter,skill_name,skill_domain,job_category,region,posting_count,avg_salary_max
0,2024,NaN,ALL,ALL,ALL,ALL,980,102487.002854
1,2024,NaN,ALL,urchade/gliner_small-v2.1,ALL,ALL,980,102487.002854
2,2024,2.0,ALL,ALL,ALL,ALL,978,102487.002854
3,2024,2.0,ALL,urchade/gliner_small-v2.1,ALL,ALL,978,102487.002854
4,2024,NaN,ALL,urchade/gliner_small-v2.1,ALL,North America,975,102469.628849
5,2024,2.0,ALL,urchade/gliner_small-v2.1,ALL,North America,975,102469.628849
6,2024,2.0,ALL,ALL,ALL,North America,975,102469.628849
7,2024,NaN,ALL,ALL,ALL,North America,975,102469.628849
11,2024,NaN,ALL,urchade/gliner_small-v2.1,Other,ALL,325,86360.464385
10,2024,NaN,ALL,ALL,Other,ALL,325,86360.464385


## 13. Performance Benchmark
Perbandingan query dengan dan tanpa Materialized View.

In [28]:
import time
# RAW QUERY (WITHOUT MATERIALIZED VIEW)
query_raw = """
    SELECT
        ds.skill_name,
        dt.week_label,

        COUNT(DISTINCT f.job_id) AS cnt

    FROM fact_job_posting f

    JOIN bridge_job_skill b
        ON f.job_id = b.job_id

    JOIN dim_skill ds
        ON b.skill_id = ds.skill_id

    JOIN dim_time dt
        ON f.time_id = dt.time_id

    GROUP BY
        ds.skill_name,
        dt.week_label

    ORDER BY cnt DESC

    LIMIT 20
"""

# MATERIALIZED VIEW QUERY
query_mv = """
    SELECT
        skill_name,
        week_label,

        SUM(posting_count) AS cnt

    FROM mv_weekly_skill_demand

    GROUP BY
        skill_name,
        week_label

    ORDER BY cnt DESC

    LIMIT 20
"""

# PERFORMANCE BENCHMARK
runs = 3

times_raw = []
times_mv  = []

for _ in range(runs):

    t0 = time.time()

    pd.read_sql(query_raw, engine)

    times_raw.append(time.time() - t0)

    t0 = time.time()

    pd.read_sql(query_mv, engine)

    times_mv.append(time.time() - t0)

avg_raw = sum(times_raw) / runs
avg_mv  = sum(times_mv) / runs

speedup = (
    avg_raw / avg_mv
    if avg_mv > 0
    else float("inf")
)

print(f"Without Materialized View : {avg_raw:.4f} sec")
print(f"With Materialized View    : {avg_mv:.4f} sec")
print(f"Performance Speedup       : {speedup:.2f}x")

Without Materialized View : 0.7309 sec
With Materialized View    : 0.6828 sec
Performance Speedup       : 1.07x


## 14. Buka Atoti Dashboard

In [29]:
print("TalentTrack Atoti DataMart READY")
print(f"  Dashboard : {session.url}")
print("  Cubes     : JobPostings | SkillDemand | SkillForecast")

TalentTrack Atoti DataMart READY
  Dashboard : http://localhost:9090
  Cubes     : JobPostings | SkillDemand | SkillForecast
